# Apache Iceberg

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/iceberg/iceberg.ipynb)

Apache Iceberg is an open table format for large analytical datasets: snapshots, schema and partition evolution, hidden partitioning, row-level deletes, and time travel over Parquet files, readable by Spark, Trino, Flink, Snowflake, and more. Spark SQL appears as fenced snippets; the runnable cells use `pyiceberg` with a local SQLite catalog and a temporary warehouse directory.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyiceberg": "pyiceberg[sql-sqlite,pyarrow,pyiceberg-core]>=0.9", "pyiceberg_core": "pyiceberg-core>=0.4", "sqlalchemy": "sqlalchemy>=2.0", "pyarrow": "pyarrow>=15"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import json
import tempfile
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.expressions import And, EqualTo, GreaterThanOrEqual, LessThan
from pyiceberg.partitioning import PartitionField, PartitionSpec
from pyiceberg.schema import Schema
from pyiceberg.transforms import BucketTransform, DayTransform
from pyiceberg.types import DoubleType, LongType, NestedField, StringType, TimestamptzType

WAREHOUSE = Path(tempfile.mkdtemp(prefix="iceberg-demo-"))
catalog = SqlCatalog("demo", uri=f"sqlite:///{WAREHOUSE}/catalog.db", warehouse=f"file://{WAREHOUSE}")
catalog.create_namespace("analytics")

BASE_TS = datetime(2026, 10, 1, tzinfo=timezone.utc)


def orders_batch(ids, day: int, amount_column: str = "amount") -> pa.Table:
    """Synthetic orders for one day; `amount_column` follows schema renames."""
    ids = list(ids)
    return pa.table({
        "order_id": pa.array(ids, pa.int64()),
        "customer_id": pa.array([f"C{i % 7}" for i in ids]),
        amount_column: pa.array([float(i) for i in ids]),
        "event_ts": pa.array([BASE_TS + timedelta(days=day, hours=i % 24) for i in ids], pa.timestamp("us", tz="UTC")),
    })


print("warehouse:", WAREHOUSE)

## What is Apache Iceberg?

Apache Iceberg is an open table format for large analytical datasets. It is not a compute engine by itself. PySpark, Spark SQL, Flink, Trino, Snowflake, and other engines use Iceberg metadata to read and write the same table consistently.

```text
PySpark job -> Iceberg catalog -> Iceberg table metadata -> Parquet files in S3/GCS/ADLS/HDFS
```

Under the hood an Iceberg table is four layers. The catalog holds one pointer per table: the location of the current metadata JSON file. That JSON file stores the schema, partition specs, sort orders, table properties, the snapshot log, and `current-snapshot-id`. Each snapshot points at a manifest list in Avro. The manifest list points at manifest files. Each manifest lists data files and delete files with their partition tuple, record count, file size, and column bounds.

Worked example. An append writes three Parquet files, one manifest, one manifest list, and a new metadata JSON, then swaps the catalog pointer. Snapshot `836...` becomes current and the previous snapshot id becomes its parent. A reader that loaded the old metadata JSON keeps seeing the old files until it refreshes. A query filtered on `order_ts` reads the manifest list, skips manifests whose partition summaries do not overlap, and opens only the matching Parquet files.

Create a table partitioned by `days(event_ts)`, append three days, and walk the metadata tree from the catalog pointer down to manifests:

In [ ]:
schema = Schema(
    NestedField(1, "order_id", LongType(), required=False),
    NestedField(2, "customer_id", StringType(), required=False),
    NestedField(3, "amount", DoubleType(), required=False),
    NestedField(4, "event_ts", TimestamptzType(), required=False),
)
spec = PartitionSpec(PartitionField(source_id=4, field_id=1000, transform=DayTransform(), name="event_ts_day"))
orders = catalog.create_table("analytics.orders", schema=schema, partition_spec=spec)

for day in range(3):
    orders.append(orders_batch(range(day * 100, day * 100 + 100), day))

metadata = json.loads(Path(orders.metadata_location.removeprefix("file://")).read_text())
snapshot = orders.current_snapshot()
manifests = snapshot.manifests(orders.io)
print("catalog pointer ->", Path(orders.metadata_location).name)
print("metadata keys:", sorted(metadata)[:8], "...")
print("current snapshot:", snapshot.snapshot_id, "-> manifest list", Path(snapshot.manifest_list).name)
print("manifests:", [Path(m.manifest_path).name for m in manifests])

assert len(orders.snapshots()) == 3
assert len(manifests) == 3  # one manifest per append; the manifest list points at all of them
assert orders.scan().to_arrow().num_rows == 300

Common mistake: treating the catalog as the table. The catalog only stores a pointer; the table state is the metadata JSON plus the manifests and data files. If you recreate the catalog entry without the original metadata location, or delete the metadata directory, the table is lost even though the Parquet files are still on storage.

## How do you enable Iceberg in PySpark?

Enabling Iceberg has two parts. The SQL extensions add Iceberg DDL and procedures, and a named catalog tells Spark where the pointer store lives. Set `spark.sql.extensions` to `org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions`, then register a catalog with `spark.sql.catalog.<name>`. The `<name>.type` value picks the implementation: `hive` for a Hive metastore, `hadoop` for a filesystem directory, `rest` for a catalog server, and `glue` for the AWS Glue Data Catalog.

Open-source Spark with an Iceberg REST catalog (in Databricks with Unity Catalog the catalog is preconfigured):

```python
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lake.type", "rest")              # or hive, glue, hadoop, jdbc
    .config("spark.sql.catalog.lake.uri", "https://catalog.example.com/iceberg")
    .config("spark.sql.catalog.lake.warehouse", "s3://acme-lake/warehouse")
    .getOrCreate()
)
```

```sql
-- Create a catalog-qualified Iceberg table. The catalog name is part of the table name.
CREATE TABLE IF NOT EXISTS prod.analytics.orders (
    order_id    BIGINT,
    customer_id BIGINT,
    order_ts    TIMESTAMP,
    status      STRING,
    amount      DECIMAL(12, 2)
)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES ('write.target-file-size-bytes' = '536870912');
```

## How do you create and write to an Iceberg table in PySpark?

```python
spark.sql("""
create table if not exists lake.analytics.orders (
  order_id bigint, customer_id string, order_ts timestamp, status string, amount decimal(12, 2)
) using iceberg
partitioned by (days(order_ts))
""")

orders_df.writeTo("lake.analytics.orders").append()

# Replace exactly one business day, atomically
(
    orders_df.where("order_ts >= '2026-10-04' and order_ts < '2026-10-05'")
    .writeTo("lake.analytics.orders")
    .overwrite(F.expr("order_ts >= '2026-10-04' and order_ts < '2026-10-05'"))
)
```

Interview point: `writeTo` is preferred over path-based writes because it writes through the Iceberg table abstraction and participates in Iceberg commits.

A write plans first. Spark resolves the base snapshot, computes partition values from the transform, writes data files, records column stats into manifests, then commits by creating a new metadata JSON and swapping the catalog pointer. No reader sees the new files until the swap succeeds. DataFrameWriterV2 (`writeTo`) follows this path; a path-based `write.parquet` does not, because it never writes Iceberg metadata.

```sql
CREATE TABLE IF NOT EXISTS prod.analytics.orders (
    order_id    BIGINT,
    customer_id BIGINT,
    order_ts    TIMESTAMP,
    status      STRING,
    amount      DECIMAL(12, 2)
)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES (
    'write.format.default' = 'parquet',
    'write.target-file-size-bytes' = '536870912'
);

INSERT INTO prod.analytics.orders
SELECT order_id, customer_id, order_ts, status, amount
FROM staging.orders_raw;
```

## How does Iceberg improve data lake table management?

Iceberg replaces directory listing with metadata planning. The engine reads one metadata JSON and one manifest list, then only the manifests and files whose bounds overlap the query. There is no recursive `LIST` over the table directory, so planning cost depends on manifest count, not on object-store listing speed.

Metadata tables expose what a planner reads. These are the pyiceberg equivalents of `orders.snapshots`, `orders.files`, and `orders.manifests` in Spark:

In [ ]:
files = orders.inspect.files()
print(files.select(["file_path", "record_count", "file_size_in_bytes"]).to_pylist()[0])
print("data files:", files.num_rows, "manifests:", orders.inspect.manifests().num_rows)
assert files.num_rows == 3 and sum(files.column("record_count").to_pylist()) == 300

## What problems does Iceberg solve compared to plain Parquet datasets?

The concrete wins are atomic commits, schema evolution by field id, partition evolution, row-level deletes, and pruning from column stats. A plain Parquet dataset has none of these: a failed write leaves partial files that readers may see, a rename breaks readers, and changing the partition layout means rewriting every file.

Worked example: an append job fails after writing 40 of 100 files. In a Parquet dataset those 40 files are already visible under the partition prefix, so a reader can see half the batch. In Iceberg nothing is committed, the reader still sees the previous snapshot, and the 40 files become orphans that `remove_orphan_files` can clean up later.

## What is hidden partitioning in Iceberg and how do you enable it in PySpark?

Hidden partitioning means users filter on logical columns, while Iceberg stores partition transforms in table metadata. You enable it with partition transforms in table DDL.

Queries filter on `event_ts` itself; Iceberg maps the predicate to `days(event_ts)` and prunes files during planning:

In [ ]:
day_two = And(
    GreaterThanOrEqual("event_ts", "2026-10-02T00:00:00+00:00"),
    LessThan("event_ts", "2026-10-03T00:00:00+00:00"),
)
planned = list(orders.scan(row_filter=day_two).plan_files())
all_files = list(orders.scan().plan_files())
print(f"files planned: {len(planned)} of {len(all_files)}")
rows = orders.scan(row_filter=day_two).to_arrow()
print("rows:", rows.num_rows)
assert len(planned) == 1 and rows.num_rows == 100

+ `days(order_ts)`: common daily reporting and backfills
+ `bucket(32, customer_id)`: frequent customer-keyed deletes, joins, or merges
+ `truncate(8, region)`: prefix-style grouping for string fields with many values

A partition transform turns a source column value into a partition value: `days(ts)` produces a date, `bucket(N, id)` produces a hash bucket from `N` and the column, and `truncate(W, s)` produces a prefix of width `W`. The transform lives in the partition spec in table metadata, so queries keep filtering the original column. Iceberg projects the filter through the transform to skip files.

## What is partition evolution and how do you change partitions in PySpark?

Partition evolution lets you change the partition spec for new data without rewriting all old files. Historical files keep their old partition spec; new files use the new spec.

In [ ]:
with orders.update_spec() as update:
    update.add_field("customer_id", BucketTransform(8), "customer_bucket")

for spec_id, partition_spec in orders.specs().items():
    print(spec_id, [f"{f.name}={f.transform}" for f in partition_spec.fields])

orders.append(orders_batch(range(300, 340), 3))
spec_ids = orders.inspect.files().column("spec_id").to_pylist()
print("spec id per data file:", spec_ids)

assert orders.spec().spec_id == 1
assert spec_ids.count(0) == 3            # old files keep the old spec: nothing was rewritten
assert spec_ids.count(1) > 1             # new files are bucketed by customer

```sql
ALTER TABLE lake.analytics.orders ADD PARTITION FIELD bucket(16, customer_id);
ALTER TABLE lake.analytics.orders DROP PARTITION FIELD days(order_ts);
-- Physically re-lay old files only if queries need it
CALL lake.system.rewrite_data_files(table => 'analytics.orders', where => "order_ts < '2026-10-01'");
```

Each partition spec has an id. Changing the spec creates a new spec id; new writes produce manifests for the new spec while old manifests keep the old spec. Planning reads every spec, so queries span both layouts. Old data files are not rewritten by the `ALTER`.

## How does schema evolution work and how do you apply it in PySpark?

Iceberg tracks columns by stable field IDs, not only names. This makes add, drop, rename, and compatible type changes safer than unmanaged Parquet.

In [ ]:
first_snapshot = orders.snapshots()[0].snapshot_id
with orders.update_schema() as update:
    update.add_column("channel", StringType())
    update.rename_column("amount", "gross_amount")

print([(f.field_id, f.name) for f in orders.schema().fields])
current = orders.scan(row_filter=EqualTo("order_id", 1)).to_arrow()
print(current.to_pylist())

# Renamed column still reads old files (matched by field id); the new column reads as null
assert current.column("gross_amount").to_pylist() == [1.0]
assert current.column("channel").to_pylist() == [None]
assert orders.schema().find_field("gross_amount").field_id == 3

```sql
ALTER TABLE lake.analytics.orders ADD COLUMN channel string;
ALTER TABLE lake.analytics.orders RENAME COLUMN amount TO gross_amount;
ALTER TABLE lake.analytics.orders ALTER COLUMN quantity TYPE bigint;   -- int -> bigint widening
ALTER TABLE lake.analytics.orders DROP COLUMN legacy_flag;
```

Every column gets a unique field id in the schema. A rename changes the name mapped to an id and is metadata-only; reads project by id, and Parquet footers store ids, so old files resolve correctly after a rename. Type changes are allowed only as safe promotions such as `int` to `long`, `float` to `double`, or a wider `decimal` precision.

Common mistake: dropping a column and re-adding it with the same name. Iceberg assigns a new field id, so old files have no value for that id and readers return null for the whole history of that field. Prefer rename when you only want to change the name.

## What is a snapshot in Iceberg and how do you inspect it in PySpark?

A snapshot is a committed table version. Every successful append, overwrite, merge, delete, or schema change creates a new table state.

In [ ]:
for entry in orders.inspect.snapshots().select(["snapshot_id", "parent_id", "operation"]).to_pylist():
    print(entry)
assert orders.inspect.snapshots().num_rows == len(orders.snapshots())

A snapshot records a manifest list, a parent snapshot id, a sequence number, a timestamp, and a summary map such as `added-data-files`, `added-records`, `total-records`, and `total-data-files`. Snapshot ids are 64-bit longs assigned at commit time, usually derived from the commit timestamp and made monotonic. The `main` branch points at exactly one current snapshot.

## What is time travel and how do you use it in PySpark?

Time travel queries a table as it existed at a previous snapshot or timestamp. Use it for debugging, audits, regression checks, and reproducing reports.

In [ ]:
as_of_first = orders.scan(snapshot_id=first_snapshot).to_arrow()
print("rows at first snapshot:", as_of_first.num_rows, "columns:", as_of_first.column_names)
print("rows now:", orders.scan().to_arrow().num_rows)
assert as_of_first.num_rows == 100
assert "amount" in as_of_first.column_names  # the snapshot's own schema

Time travel resolves a version to a snapshot: by id, by timestamp (the latest snapshot committed at or before that time on an ancestor path), or by branch or tag name. Iceberg then plans the scan against that snapshot's manifests. In Spark SQL use `VERSION AS OF` and `TIMESTAMP AS OF`; in the DataFrame reader use `snapshot-id`, `as-of-timestamp`, `branch`, or `tag`.

```sql
-- Snapshot id from the snapshots metadata table.
SELECT * FROM prod.analytics.orders VERSION AS OF 8364525912309844123;

-- A wall-clock time: the latest snapshot at or before this timestamp.
SELECT * FROM prod.analytics.orders TIMESTAMP AS OF '2026-10-04 12:00:00';

-- A named branch or tag.
SELECT * FROM prod.analytics.orders VERSION AS OF 'finance_close_2026_09';
```

Common mistake: assuming any past timestamp is queryable. Time travel only reaches snapshots that still exist, and `expire_snapshots` removes the rest. A query for an expired snapshot fails; it does not silently fall back to current data.

## How does Iceberg handle concurrent writes?

Iceberg uses optimistic concurrency control. A writer plans changes from a base snapshot, writes new files, then tries to commit new metadata. If another writer committed a conflicting change first, the commit fails or retries instead of silently corrupting the table.

Commits are optimistic. A writer reads the current metadata, creates new files and a candidate metadata JSON, then asks the catalog for an atomic pointer swap. If the pointer moved, the catalog rejects the commit and the writer validates whether the concurrent change is compatible. Append-style commits usually rebase and retry; conflicting isolation-level changes fail. Retries are governed by `commit.retry.num-retries`, which defaults to 4.

Two writers load the same base metadata. The first commits; the second's compare-and-swap on the catalog pointer fails, so pyiceberg refreshes and retries (the log line shows it). Both appends land because appends do not conflict:

In [ ]:
writer_a = catalog.load_table("analytics.orders")
writer_b = catalog.load_table("analytics.orders")
base_snapshot = writer_a.current_snapshot().snapshot_id

writer_a.append(orders_batch(range(400, 405), 4, "gross_amount").append_column("channel", pa.array(["web"] * 5)))
writer_b.append(orders_batch(range(405, 410), 4, "gross_amount").append_column("channel", pa.array(["app"] * 5)))

orders.refresh()
new_snapshots = [s for s in orders.snapshots() if s.parent_snapshot_id is not None][-2:]
print("parents of the last two snapshots:", [s.parent_snapshot_id for s in new_snapshots])
assert orders.scan(row_filter=GreaterThanOrEqual("order_id", 400)).to_arrow().num_rows == 10
assert new_snapshots[0].parent_snapshot_id == base_snapshot  # writer_b was rebased on top of writer_a

## What are equality deletes and positional deletes, and how do you use deletes in PySpark?

Iceberg supports row-level deletes by writing delete files instead of immediately rewriting every affected data file.

+ Equality deletes mark rows as deleted based on column values, such as `order_id = 1001`.
+ Positional deletes mark specific row positions in specific data files.

In v2 tables Iceberg adds sequence numbers. A delete file applies only to data files with a lower sequence number, so a later insert of the same key survives. Positional deletes store `(file_path, row_position)` pairs and are usually produced by `MERGE`; equality deletes store the deleted key values and are common with Flink. With `write.merge.mode=merge-on-read` the engine writes delete files instead of rewriting data files, and readers merge them at scan time.

In [ ]:
before = orders.scan().to_arrow().num_rows
before_snapshot = orders.current_snapshot().snapshot_id
orders.delete(EqualTo("customer_id", "C3"))   # pyiceberg rewrites affected files (copy-on-write)
after = orders.scan().to_arrow().num_rows
print(f"rows {before} -> {after}; operation:", orders.current_snapshot().summary.operation)
assert orders.scan(row_filter=EqualTo("customer_id", "C3")).to_arrow().num_rows == 0
assert orders.scan(snapshot_id=before_snapshot).to_arrow().num_rows == before  # old snapshot still readable

```sql
DELETE FROM lake.analytics.orders WHERE customer_id = 'C123';
-- Merge-on-read: write delete files instead of rewriting data files
ALTER TABLE lake.analytics.orders SET TBLPROPERTIES (
  'write.delete.mode' = 'merge-on-read',
  'write.update.mode' = 'merge-on-read',
  'write.merge.mode'  = 'merge-on-read'
);
```

## How do upserts and merges work with Iceberg in PySpark?

Use `MERGE INTO` from PySpark through `spark.sql`. Deduplicate CDC input first so each target key has one winning source record.

```sql
MERGE INTO lake.analytics.orders AS t
USING orders_cdc_deduped AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.op = 'D' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT *;
```

pyiceberg's `upsert` performs the same matched-update / not-matched-insert on a join key:

In [ ]:
changes = pa.table({
    "order_id": pa.array([1, 999], pa.int64()),
    "customer_id": ["C1", "C9"],
    "gross_amount": [111.0, 999.0],
    "event_ts": pa.array([BASE_TS, BASE_TS], pa.timestamp("us", tz="UTC")),
    "channel": ["web", "app"],
})
result = orders.upsert(changes, join_cols=["order_id"])
print(result)
updated = orders.scan(row_filter=EqualTo("order_id", 1)).to_arrow().to_pylist()[0]
assert (result.rows_updated, result.rows_inserted) == (1, 1)
assert updated["gross_amount"] == 111.0

A merge is a join on the merge key. Iceberg plans the target files that can contain matching keys, shuffles source and target by key, applies the matched and not-matched actions, and commits one snapshot. Partition pruning on the merge key limits how many target files are rewritten. `write.merge.mode` selects copy-on-write (rewrite touched files) or merge-on-read (write delete files plus inserts).

## Why does the small files problem happen and how do you mitigate it in PySpark?

Small files happen when many jobs write small batches, streaming jobs commit frequently, partitioning is too granular, or Spark uses too many writer tasks.

Every commit adds at least one manifest and one data file per writing task. Small files hurt twice: more manifests slow planning, and more files raise per-file open cost. The default `write.target-file-size-bytes` is 512 MB and `read.split.open-file-cost` is 4 MB, so a table of 1 KB files pays an open cost far larger than the data it reads.

In [ ]:
# A file-size histogram is the quickest read on small-file growth.
files = [
    {"path": "data/00000.parquet", "size_mb": 1.2},
    {"path": "data/00001.parquet", "size_mb": 3.4},
    {"path": "data/00002.parquet", "size_mb": 480.0},
    {"path": "data/00003.parquet", "size_mb": 512.0},
]

tiny = [f for f in files if f["size_mb"] < 128]
total_mb = sum(f["size_mb"] for f in files)
print(f"{len(tiny)} of {len(files)} files are under 128 MB")
print(f"table size = {total_mb:.1f} MB")
assert len(tiny) == 2 and round(total_mb, 1) == 996.6

```sql
ALTER TABLE lake.analytics.orders SET TBLPROPERTIES ('write.target-file-size-bytes' = '536870912');
CALL lake.system.rewrite_data_files(table => 'analytics.orders', options => map('min-input-files', '5'));
```

## What maintenance operations are common for Iceberg tables in PySpark?

Iceberg tables need periodic maintenance after streaming writes, merges, deletes, and frequent schema/partition changes.

Each procedure has a narrow job. `rewrite_data_files` bin-packs small and delete-heavy files into target-size files. `rewrite_manifests` re-clusters manifest entries by partition to speed planning. `expire_snapshots` deletes snapshots and the files only they reference, with defaults of `older_than` five days ago and `retain_last` 1. `remove_orphan_files` deletes files no snapshot references, defaulting to older than three days ago.

```sql
-- Compact: bin-pack small files and materialize merge-on-read deletes.
CALL prod.system.rewrite_data_files(
  table => 'analytics.orders',
  options => map('target-file-size-bytes', '536870912', 'min-input-files', '5')
);

-- Re-cluster manifests so planning can skip more of them.
CALL prod.system.rewrite_manifests(table => 'analytics.orders', sort_by => array('order_ts_day'));

-- Retire snapshots past the audit window but always keep the last 10.
CALL prod.system.expire_snapshots(
  table => 'analytics.orders',
  older_than => TIMESTAMP '2026-09-01 00:00:00',
  retain_last => 10
);

-- Inspect orphan candidates first.
CALL prod.system.remove_orphan_files(
  table => 'analytics.orders',
  older_than => TIMESTAMP '2026-09-01 00:00:00',
  dry_run => true
);
```

Snapshot expiry with pyiceberg: expired snapshots are no longer reachable for time travel:

In [ ]:
snapshot_count = len(orders.snapshots())
current_id = orders.current_snapshot().snapshot_id
cutoff_ms = max(s.timestamp_ms for s in orders.snapshots()) + 1
orders.maintenance.expire_snapshots().older_than(datetime.fromtimestamp(cutoff_ms / 1000, tz=timezone.utc)).commit()
orders.refresh()
print(f"snapshots: {snapshot_count} -> {len(orders.snapshots())}")
assert [s.snapshot_id for s in orders.snapshots()] == [current_id]  # the current snapshot is always kept

Common mistake: running `remove_orphan_files` with a cutoff inside the window of an in-flight job. A job that has written files but not yet committed will see its files deleted. Set `older_than` beyond the longest job and dry-run first.

## What is an Iceberg catalog and how do you configure one for PySpark?

An Iceberg catalog stores namespaces and the pointer to each table's current metadata file. It is how PySpark finds and commits to tables.

Other catalog choices still exist, such as Hive metastore, AWS Glue, Apache Polaris, Hadoop catalog, REST catalog, and Nessie. But if your platform is Databricks-centered, use Unity Catalog first because it centralizes governance, permissions, lineage, and external Iceberg REST access.

The catalog is the concurrency control point. Hive uses a metastore transaction, REST uses the catalog server's compare-and-swap, Glue uses `UpdateTable` with a version check, and Hadoop uses atomic rename plus a `version-hint.text` file. The REST protocol is what lets engines with no shared library, such as Snowflake or Trino, commit under the same rules.

```sql
CREATE NAMESPACE IF NOT EXISTS prod.analytics;

CREATE TABLE IF NOT EXISTS prod.analytics.orders (
    order_id BIGINT,
    order_ts TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(order_ts));
```

## How would you migrate an existing Parquet dataset to Iceberg with PySpark?

The safest approach is usually to create a new Iceberg table and rewrite the data into it. This lets you enforce schema, clean bad records, and choose partitioning deliberately.

There are three paths. Rewriting through `INSERT` gives the cleanest metadata and partition layout. `add_files` registers existing files without moving them, but it does not validate the schema and Iceberg then owns those files. `migrate` replaces an existing Hive or Spark table with an Iceberg table that keeps the source data files; `snapshot` makes a lightweight test copy that shares the same files.

```sql
-- In-place migration: the source table is retained as a backup by default.
CALL prod.system.migrate('spark_catalog.analytics.orders_parquet');

-- Test first with a lightweight snapshot table that shares the same files.
CALL prod.system.snapshot(
  source_table => 'spark_catalog.analytics.orders_parquet',
  table => 'analytics.orders_snapshot'
);

-- Register existing Parquet files as Iceberg data files without moving them.
CALL prod.system.add_files(
  table => 'analytics.orders',
  source_table => '`parquet`.`s3://company-lake/legacy/orders`'
);

-- Register a metadata JSON that already exists on storage.
CALL prod.system.register_table(
  table => 'analytics.orders',
  metadata_file => 's3://company-lake/warehouse/analytics/orders/metadata/00001-abc.metadata.json'
);
```

## Iceberg vs Delta Lake: what is the difference with examples?

Iceberg and Delta Lake both make data lake tables more reliable than plain Parquet, but they make different ecosystem and interoperability trade-offs.

+ Iceberg focuses on open, multi-engine table interoperability with hidden partitioning and strong partition evolution.
+ Delta Lake is deeply integrated with the Delta/Databricks ecosystem and has mature Delta-specific optimization features.
+ Iceberg partitioning is metadata-driven, so users can filter `order_ts` while Iceberg prunes `days(order_ts)`.
+ Traditional Delta partitioning often exposes physical partition columns like `order_date`, though Delta UniForm can expose Delta tables to Iceberg readers.

The metadata design is the difference:
+ Iceberg keeps a metadata JSON, a manifest list, and manifest files, and stores partition transforms in the spec.
+ Delta keeps a `_delta_log` of JSON commits with periodic Parquet checkpoints, and traditional Delta partitioning uses physical partition columns.
+ Iceberg is designed around the REST catalog protocol so many engines can commit; Delta is designed around the Delta protocol and the Databricks ecosystem.
+ Deletes: Iceberg writes positional or equality delete files; Delta writes deletion vectors.

## How do you make a Delta table Iceberg-compatible?

The common modern approach is Delta UniForm, which lets a Delta table generate Iceberg-compatible metadata so Iceberg readers can read it. The Delta table remains a Delta table; Iceberg clients should treat it as read-only unless your platform explicitly supports more.

```sql
ALTER TABLE main.silver.orders SET TBLPROPERTIES (
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
);
```

UniForm generates metadata, it does not convert the table. Delta stays the source of truth and keeps writing `_delta_log`; a background process writes Iceberg metadata beside it that points at the same Parquet files. Iceberg readers see snapshots that track Delta commits. Supported input formats and table features are restricted, so check the compatibility list before enabling it.

## How do you access a PySpark-created Iceberg table in Snowflake?

1. Databricks PySpark writes an Iceberg table registered in Unity Catalog.
2. A Databricks service principal is granted access to the Unity Catalog catalog/schema/table.
3. Snowflake creates an Iceberg REST catalog integration that points to the Databricks Unity Catalog Iceberg REST endpoint.
4. Snowflake creates a catalog-linked database that syncs Unity Catalog namespaces and Iceberg tables.
5. Snowflake queries the Iceberg table without copying the data.

```sql
GRANT USE CATALOG ON CATALOG main TO `<application_id>`;
GRANT USE SCHEMA ON SCHEMA main.analytics TO `<application_id>`;
GRANT EXTERNAL USE SCHEMA ON SCHEMA main.analytics TO `<application_id>`;
GRANT SELECT ON TABLE main.analytics.orders TO `<application_id>`;
```

```sql
CREATE OR REPLACE CATALOG INTEGRATION databricks_uc_iceberg
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  REST_CONFIG = (
    CATALOG_URI = 'https://<databricks-workspace-url>/api/2.1/unity-catalog/iceberg-rest'
    CATALOG_NAME = 'main'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = OAUTH
    OAUTH_TOKEN_URI = 'https://<databricks-workspace-url>/oidc/v1/token'
    OAUTH_CLIENT_ID = '<databricks-service-principal-application-id>'
    OAUTH_CLIENT_SECRET = '<databricks-service-principal-secret>'
    OAUTH_ALLOWED_SCOPES = ('all-apis')
  )
  ENABLED = TRUE;
```

```sql
CREATE DATABASE databricks_uc
  LINKED_CATALOG = (
    CATALOG = 'databricks_uc_iceberg'
    BLOCKED_NAMESPACES = ('information_schema')
  );
```

Snowflake acts as an Iceberg REST client. A catalog integration names the REST endpoint and the OAuth credentials; a catalog-linked database syncs namespaces and tables. With `ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS`, the catalog hands Snowflake short-lived storage credentials, so no long-lived cloud key is stored in Snowflake.

## Iceberg vs Delta vs Hudi: when would you choose Iceberg?

Choose Iceberg when you need an open, engine-neutral table format with strong metadata handling, hidden partitioning, partition evolution, schema evolution, time travel, and multi-engine access.

+ Iceberg: multi-engine lakehouse, open table standard, hidden partitioning, partition/schema evolution
+ Delta: strong fit in Delta/Databricks-centered environments, with UniForm when external Iceberg readers are needed
+ Hudi: strong fit for CDC-heavy ingestion and incremental pull patterns

Each format optimizes for a different core abstraction. Hudi is built around a timeline and copy-on-write or merge-on-read table types, tuned for upsert-heavy ingestion and incremental pull. Delta is built around the transaction log and tight Databricks integration. Iceberg is built around the four-layer metadata tree and the REST catalog protocol, tuned for many engines reading and writing one table.

## How do branching and tagging work in Iceberg?

A branch is a named mutable reference to a snapshot; a tag is a named immutable reference. Both live in the `refs` map in table metadata. `main` is the default branch and never expires. Writes can target a branch for a write-audit-publish flow: write to an audit branch, validate, then fast-forward `main`.

In [ ]:
latest = orders.current_snapshot().snapshot_id
orders.manage_snapshots().create_tag(latest, "eod-2026-10-04").create_branch(latest, "audit").commit()
refs = {name: ref.snapshot_ref_type.value for name, ref in orders.metadata.refs.items()}
print(refs)
assert refs == {"main": "branch", "eod-2026-10-04": "tag", "audit": "branch"}

```sql
-- Tag a snapshot for an auditable report.
ALTER TABLE prod.analytics.orders
  CREATE TAG finance_close_2026_09 AS OF VERSION 8364525912309844123 RETAIN 180 DAYS;

-- Branch for write-audit-publish.
ALTER TABLE prod.analytics.orders
  CREATE BRANCH audit_2026_10_04 RETAIN 7 DAYS WITH SNAPSHOT RETENTION 2 SNAPSHOTS;

-- Write to and read from the branch.
INSERT INTO prod.analytics.orders.branch_audit_2026_10_04
VALUES (1001, 42, TIMESTAMP '2026-10-04 09:00:00', 'new', 19.99);

SELECT * FROM prod.analytics.orders VERSION AS OF 'audit_2026_10_04';

-- Inspect references.
SELECT * FROM prod.analytics.orders.refs;
```

## How do you tune Iceberg tables for performance?

Tune in this order: prune more, then read fewer and larger files, then plan faster. Pruning depends on the partition spec and column stats. File efficiency depends on the size histogram. Planning depends on manifest count. Delete-file accumulation and catalog overhead come last. Every one of these is measurable from the metadata tables before you change anything.

The numbers to collect: `files` gives `file_size_in_bytes` and `content` (0 data, 1 position deletes, 2 equality deletes); `manifests` gives manifest count and per-manifest file counts; `snapshots` gives commit age, operation, and summary counts; `partitions` gives file and record counts per partition. A healthy table has files near `write.target-file-size-bytes`, few delete files, and manifest counts in the hundreds, not tens of thousands.

```sql
-- File-size histogram: how much of the table is in small files?
SELECT CASE
         WHEN file_size_in_bytes < 16 * 1024 * 1024 THEN 'tiny (<16MB)'
         WHEN file_size_in_bytes < 128 * 1024 * 1024 THEN 'small (<128MB)'
         WHEN file_size_in_bytes < 640 * 1024 * 1024 THEN 'target'
         ELSE 'large'
       END AS size_bucket,
       COUNT(*) AS files,
       ROUND(SUM(file_size_in_bytes) / POWER(1024, 3), 1) AS gib
FROM prod.analytics.orders.files
WHERE content = 0
GROUP BY 1;

-- Delete-file accumulation from merge-on-read writes.
SELECT content, COUNT(*) AS files, ROUND(AVG(record_count), 0) AS avg_rows
FROM prod.analytics.orders.files
WHERE content > 0
GROUP BY 1;

-- Planning pressure: manifests and the files they track.
SELECT COUNT(*) AS manifests,
       SUM(added_data_files_count + existing_data_files_count) AS tracked_files
FROM prod.analytics.orders.manifests;

-- Snapshot age and operation, to size the expiry window.
SELECT committed_at, snapshot_id, operation,
       summary['total-data-files'] AS total_data_files
FROM prod.analytics.orders.snapshots
ORDER BY committed_at DESC;
```

## How do you choose a partition transform?

Match the transform to the dominant predicate. Date-range filters want a time transform. High-cardinality equality filters and keyed merges want `bucket`. Low-cardinality category filters want `identity` or `truncate`. Aim for roughly one to a few million rows per partition: enough files to prune, few enough partitions to avoid a metadata explosion.

Hidden partitioning means the query never mentions the partition column, so the transform can change later. Prefer one time transform plus at most one `bucket`; each extra field multiplies the partition count and the manifest count. `truncate(8, region)` splits long strings into prefixes, while `bucket(32, customer_id)` spreads customers evenly and tolerates skew.

## How do you keep manifests and metadata small?

Scan planning cost grows with manifest count, and manifests grow with every commit. `rewrite_manifests` rewrites them clustered by partition so planning can skip whole manifests. `sort_by` clusters manifests by the partition field your queries filter on most. The goal is manifests near `read.split.metadata-target-size` (32 MB), not thousands of tiny ones.

Metadata JSON files also accumulate. `write.metadata.previous-versions-max` defaults to 100 and caps how many are tracked; `write.metadata.delete-after-commit.enabled` defaults to false and deletes the oldest tracked versions after each commit. Turn both on when commit rate is high, but keep a window long enough for rollback. `read.split.planning-lookback` defaults to 10.

```sql
-- Cap metadata history and let commits delete old metadata JSON.
ALTER TABLE prod.analytics.orders SET TBLPROPERTIES (
  'write.metadata.previous-versions-max' = '50',
  'write.metadata.delete-after-commit.enabled' = 'true'
);

-- Cluster manifests by the partition field used in most filters.
CALL prod.system.rewrite_manifests(
  table => 'analytics.orders',
  sort_by => array('order_ts_day')
);
```

## Real world use cases for Apache Iceberg

### 1. Multi-engine lakehouse on object storage

A retailer keeps one copy of curated data in S3 as Iceberg. Spark writes it, Trino serves ad-hoc analytics, Athena powers a serverless dashboard, and Snowflake reads selected tables for finance - all against the same files and the same snapshot.

Why Iceberg: hidden partitioning means readers do not need to know the partition layout, and partition evolution lets the layout change without rewriting history. A single format removes the copy-per-engine pattern.

### 2. Streaming CDC with upserts and deletes

Flink or Spark Structured Streaming consumes Debezium events and applies them to Iceberg with `MERGE`. Equality deletes let a streaming job mark rows deleted without rewriting whole partitions.

```sql
MERGE INTO lakehouse.customers AS t
USING cdc_batch AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.op = 'd' THEN DELETE
WHEN MATCHED THEN UPDATE SET name = s.name, email = s.email, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, name, email, updated_at)
VALUES (s.customer_id, s.name, s.email, s.updated_at);
```

Why Iceberg: row-level deletes and ACID commits make the lakehouse table behave like a mutable database table without nightly full rebuilds.

### 3. GDPR deletion and right-to-be-forgotten

A deletion request must remove one customer's rows from a 500 TB table. Rewriting every partition is unacceptable, so the pipeline deletes by predicate and then expires the old snapshots that still contain the data.

Why Iceberg: predicate deletes plus snapshot expiry give an auditable path to true removal, which copy-on-write-parquet pipelines cannot offer without rewriting everything.

### 4. Reproducible reporting and incident recovery

A finance report must be reproducible as of its publication date. Each successful load tags a snapshot; the reporting job pins that snapshot, and a bad deployment is rolled back by pointing the table at the previous snapshot.

Why Iceberg: snapshot isolation gives point-in-time reads and fast rollback that do not depend on restoring backups.

### 5. Migration from Hive and Parquet with a zero-downtime cutover

A team migrates partitioned Parquet tables in place, keeping the same storage paths, and dual-runs readers during a validation window.

```sql
-- register existing Parquet files as an Iceberg table without moving data
CALL prod.system.migrate('analytics.orders_parquet');
```

Why Iceberg: in-place migration avoids a second copy, and snapshot isolation makes the validation window safe for existing readers.

## Iceberg exercises

The exercises model the arithmetic that Iceberg procedures use. Compaction output is `ceil(total_bytes / target_file_size)`, the rewrite trigger uses the `delete-ratio-threshold` default of 0.3, and snapshot expiry combines an age rule with `retain_last`. Treat the asserted numbers as a check on the formula, not as a prediction of a real table.

### Try it 1. Choose a partition transform

**Task.** For each query pattern, choose a transform: `days` for a timestamp range filter, `identity` for a low-cardinality equality filter, and `bucket(64, ...)` for a high-cardinality equality filter.

**Hint.** Compare the column cardinality with a threshold of 1000 to separate low from high cardinality, and treat the timestamp column as a range predicate.

**Expected result.** `{'order_ts': 'days(order_ts)', 'customer_id': 'bucket(64, customer_id)', 'region': 'identity(region)'}`.

In [ ]:
patterns = {
    "order_ts": ("range", 90),
    "customer_id": ("equality", 2_000_000),
    "region": ("equality", 40),
}


def choose_transform(column, kind, cardinality):
    if kind == "range":
        return f"days({column})"
    if cardinality <= 1000:
        return f"identity({column})"
    return f"bucket(64, {column})"


transforms = {
    col: choose_transform(col, kind, card)
    for col, (kind, card) in patterns.items()
}
for col, transform in transforms.items():
    print(f"{col:12s} -> {transform}")

assert transforms["order_ts"] == "days(order_ts)"
assert transforms["customer_id"] == "bucket(64, customer_id)"
assert transforms["region"] == "identity(region)"

### Try it 2. Plan a compaction run from a file-size histogram

**Task.** Given a histogram of file sizes and counts, count the files under 128 MB and compute how many target-size files a bin-packing rewrite produces at a 512 MB target.

**Hint.** `small = count where size < 128 MB`; `output_files = ceil(total_bytes / target_bytes)`.

**Expected result.** `5500` small files and `53` output files.

In [ ]:
import math

MB = 1024 ** 2
# size in bytes -> number of files
histogram = {
    1 * 1024: 4000,          # 1 KB
    8 * MB: 1500,            # 8 MB
    300 * MB: 50,            # 300 MB
}
target_bytes = 512 * MB

small_files = sum(count for size, count in histogram.items() if size < 128 * MB)
total_bytes = sum(size * count for size, count in histogram.items())
output_files = math.ceil(total_bytes / target_bytes)

print(f"small files (<128MB) = {small_files}")
print(f"total bytes = {total_bytes / (1024 ** 3):.2f} GiB")
print(f"compaction output files at 512MB target = {output_files}")

assert small_files == 5500
assert output_files == 53

### Try it 3. Compute snapshot expiry retention

**Task.** Given snapshots one day apart, a policy of `older_than` 7 days and `retain_last` 10, compute how many snapshots survive. A snapshot survives if it is within the age window or among the most recent `retain_last`, and the current snapshot always survives.

**Hint.** Build the retained set as the union of `days_ago <= 7` and the first 10 snapshots in newest-first order.

**Expected result.** `10` snapshots survive and `15` are expired.

In [ ]:
# newest first: (snapshot_id, days_ago)
snapshots = [(1000 - i, i) for i in range(25)]
current_id = snapshots[0][0]
older_than_days = 7
retain_last = 10

recent_ids = {sid for sid, _ in snapshots[:retain_last]}
young_ids = {sid for sid, days in snapshots if days <= older_than_days}
retained = recent_ids | young_ids | {current_id}
expired = [sid for sid, _ in snapshots if sid not in retained]

print(f"retained = {len(retained)}")
print(f"expired  = {len(expired)}")

assert len(retained) == 10
assert len(expired) == 15
assert current_id in retained

### Try it 4. Detect small-file growth from manifest counts

**Task.** Given each day's new files and total bytes, compute the average file size and flag days where it drops below half the 256 MB target.

**Hint.** `average_mb = bytes / files`; compare with `0.5 * target_mb`.

**Expected result.** The degraded days are `['2026-10-03', '2026-10-04']`.

In [ ]:
target_mb = 256
daily = {
    "2026-10-01": {"files": 100, "bytes": 100 * target_mb},
    "2026-10-02": {"files": 200, "bytes": 200 * target_mb},
    "2026-10-03": {"files": 900, "bytes": 300 * target_mb},
    "2026-10-04": {"files": 2000, "bytes": 400 * target_mb},
}

degraded = []
for day, stats in daily.items():
    average_mb = stats["bytes"] / stats["files"]
    print(f"{day}: {stats['files']:>5} files, avg {average_mb:6.1f} MB")
    if average_mb < 0.5 * target_mb:
        degraded.append(day)

print("degraded days:", degraded)
assert degraded == ["2026-10-03", "2026-10-04"]

### Try it 5. Estimate planning cost from manifest count

**Task.** Estimate scan-planning time for a table with 40 manifests and for one with 4000, using 64 MB/s metadata read throughput, 2 MB per manifest, and 1.5 ms to parse each manifest. Decide whether a manifest rewrite is warranted when planning exceeds 10 seconds.

**Hint.** `seconds = manifests * size_mb / throughput_mb_s + manifests * parse_ms / 1000`.

**Expected result.** `1.31` seconds for 40 manifests, `131.0` seconds for 4000, and the verdict `rewrite manifests`.

In [ ]:
THROUGHPUT_MB_S = 64
MANIFEST_MB = 2
PARSE_MS = 1.5


def planning_seconds(manifests):
    read_s = manifests * MANIFEST_MB / THROUGHPUT_MB_S
    parse_s = manifests * PARSE_MS / 1000
    return read_s + parse_s


small = planning_seconds(40)
large = planning_seconds(4000)
verdict = "rewrite manifests" if large > 10 else "leave manifests"

print(f"40 manifests   -> {small:.2f} s")
print(f"4000 manifests -> {large:.1f} s")
print("verdict:", verdict)

assert round(small, 2) == 1.31
assert round(large, 1) == 131.0
assert verdict == "rewrite manifests"

### Try it 6. Simulate a snapshot-based rollback

**Task.** Given a parent chain of snapshots and a current snapshot, verify that a target is an ancestor, roll back to it, and show that the snapshots after it are no longer ancestors.

**Hint.** Walk the chain from any snapshot to its parent until you reach `None`.

**Expected result.** Snapshot `3` is an ancestor, the rollback makes it current, and `4` and `5` are no longer ancestors.

In [ ]:
# snapshot id -> parent snapshot id
parents = {1: None, 2: 1, 3: 2, 4: 3, 5: 4}
current = 5


def ancestors(snapshot_id):
    chain = []
    while snapshot_id is not None:
        chain.append(snapshot_id)
        snapshot_id = parents[snapshot_id]
    return chain


target = 3
print("ancestors of current:", ancestors(current))
assert target in ancestors(current)

current = target  # rollback_to_snapshot
new_snapshot = 6
parents[new_snapshot] = current  # the next append builds on the rolled-back state

print("current after rollback:", current)
print("ancestors after rollback:", ancestors(new_snapshot))
assert current == 3
assert 4 not in ancestors(new_snapshot) and 5 not in ancestors(new_snapshot)

### Try it 7. Decide merge-on-read compaction from delete ratios

**Task.** Using the `delete-ratio-threshold` default of 0.3, find the file groups that a `rewrite_data_files` run would rewrite, and total the delete files it would remove.

**Hint.** Rewrite when `delete_ratio >= 0.3`; sum `delete_files` for the selected groups.

**Expected result.** `3` groups rewritten and `170` delete files removed.

In [ ]:
DELETE_RATIO_THRESHOLD = 0.3

groups = [
    {"partition": "p1", "delete_ratio": 0.05, "delete_files": 2},
    {"partition": "p2", "delete_ratio": 0.31, "delete_files": 40},
    {"partition": "p3", "delete_ratio": 0.30, "delete_files": 10},
    {"partition": "p4", "delete_ratio": 0.60, "delete_files": 120},
]

to_rewrite = [g for g in groups if g["delete_ratio"] >= DELETE_RATIO_THRESHOLD]
deletes_removed = sum(g["delete_files"] for g in to_rewrite)

for group in groups:
    flag = "rewrite" if group in to_rewrite else "keep"
    print(f"{group['partition']}: ratio {group['delete_ratio']:.2f} -> {flag}")

print(f"groups rewritten = {len(to_rewrite)}, delete files removed = {deletes_removed}")
assert len(to_rewrite) == 3
assert deletes_removed == 170